In [1]:
import pandas as pd
import numpy as np

In [2]:
AMAZON_PATH = '../data/cleaned/amazon_sales_clean.parquet'
STOCK_PATH = '../data/cleaned/stock_clean.parquet'
FEATURE_PATH = '../data/features/'

In [3]:
amazon = pd.read_parquet(AMAZON_PATH)
stock = pd.read_parquet(STOCK_PATH)

In [4]:
amazon['year'] = amazon['date'].dt.year
amazon['month'] = amazon['date'].dt.month

amazon['is_fulfilled_by_amazon'] = (amazon['fulfilment'] == 'Amazon').astype('int8')

amazon.rename(columns={'amount' : 'revenue'}, inplace=True)

monthly = (
    amazon
    .groupby(['sku', 'year', 'month'], as_index=False)
    .agg({
        'qty': 'sum',
        'revenue': 'sum',
        'is_fulfilled_by_amazon': 'max',
        'category': 'first',
        'size': 'first'
    })
)

monthly.to_csv(FEATURE_PATH + 'amazon_features.csv', index=False)
monthly.to_parquet(FEATURE_PATH + 'amazon_features.parquet', index=False)

In [5]:
stock_features = stock.groupby('sku', as_index=False)['stock'].sum()
stock_features.rename(columns={'stock':'stock_sum'}, inplace=True)
stock_features['stock_sum'] = stock_features['stock_sum'].astype('int32')

stock_features.to_csv(FEATURE_PATH + 'stock_features.csv', index=False)
stock_features.to_parquet(FEATURE_PATH + 'stock_features.parquet', index=False)

In [6]:
df_ml = monthly.merge(stock_features, on='sku', how='left')

df_ml['stock_sum'] = df_ml['stock_sum'].fillna(0)

df_ml['in_stock'] = (df_ml['stock_sum'] > 0).astype('int8')
df_ml['stock_log'] = np.log1p(df_ml['stock_sum'])

cat_mean = df_ml.groupby('category', observed=True)['qty'].mean()
df_ml['category_avg_qty'] = df_ml['category'].map(cat_mean)

df_ml.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16200 entries, 0 to 16199
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   sku                     16200 non-null  string  
 1   year                    16200 non-null  int32   
 2   month                   16200 non-null  int32   
 3   qty                     16200 non-null  int64   
 4   revenue                 16200 non-null  float64 
 5   is_fulfilled_by_amazon  16200 non-null  int8    
 6   category                16200 non-null  category
 7   size                    16200 non-null  category
 8   stock_sum               16200 non-null  float64 
 9   in_stock                16200 non-null  int8    
 10  stock_log               16200 non-null  float64 
 11  category_avg_qty        16200 non-null  category
dtypes: category(3), float64(3), int32(2), int64(1), int8(2), string(1)
memory usage: 839.7 KB


In [7]:
df_ml.to_csv(FEATURE_PATH + 'ml_dataset.csv', index=False)
df_ml.to_parquet(FEATURE_PATH + 'ml_dataset.parquet', index=False)